# 06 — Evaluation, Explainability & the ₹1,000 Crore Allocation

**NovaFin Group Capstone · ePGD MLDS, IIIT Bombay · Group 2 · Dipesh Kumar Yadav**

Where every module's number becomes a decision.

1. SHAP across the modules, and model cards for governance
2. Calibration — because ECL multiplies a *probability* by money
3. The fraud holdout, **scored once**, after every tuning decision
4. VaR and expected shortfall, backtested with the Kupiec test
5. Portfolio backtest: ML vs equal-weight vs Markowitz
6. Liquidity stress testing
7. **Module 11 — the integrated ₹1,000 crore allocation**

> Prerequisites: notebooks `00`–`05`.

## 1 · Preamble

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "42"

IN_COLAB = "google.colab" in str(get_ipython())  # noqa: F821
if IN_COLAB:
    import subprocess, sys
    from pathlib import Path

    REPO_URL = "https://github.com/yadavdipesh/novafin-capstone.git"
    if not Path("/content/novafin-capstone").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/novafin-capstone"], check=True)
    os.chdir("/content/novafin-capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["NOVAFIN_DATA_RAW"] = (
        "/content/drive/MyDrive/ePGD - MLDS IIT Bombay/C5 ML In Finanace/Data"
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from novafin import viz
from novafin.config import load_config
from novafin.data import load_all, make_feature_frame, make_splitter, holdout_by_time
from novafin.evaluate import (
    average_precision, calibration_table, expected_calibration_error,
    expected_credit_loss, gains_table, kupiec_pof_test, ks_statistic,
    optimal_threshold, population_stability_index, roc_auc,
)
from novafin.explain import build_model_card, compute_shap, global_importance, explain_instance
from novafin.features import build_features
from novafin.finance import (
    allocate_capital, allocation_narrative, backtest_portfolio, black_scholes_greeks,
    build_bucket_inputs, credit_decision, equal_weight, expected_shortfall,
    historical_var, markowitz_weights, monte_carlo_var, parametric_var,
    portfolio_metrics, prioritise_customers, sensitivity_analysis, stress_test_liquidity,
)
from novafin.models import cross_validate_model, evaluate_cv, load_model_specs
from novafin.tracking import ExperimentTracker
from novafin.utils.logging_utils import setup_logging
from novafin.utils.seed import seed_everything
from novafin.utils.theme import apply_theme, color, save_figure, semantic_color

cfg = load_config()
setup_logging("WARNING", log_file=cfg.paths.logs / "06_evaluation.log")
seed_everything(cfg.reproducibility.seed)
apply_theme()
pd.set_option("display.width", 200)
tracker = ExperimentTracker(cfg)
print("config fingerprint:", cfg.fingerprint())

In [ ]:
results = load_all(cfg=cfg)
features, matrices = {}, {}
for key, loaded in results.items():
    built = build_features(key, loaded.frame, cfg)
    features[key] = built
    spec = cfg.dataset(key)
    extra = [c for c in ("fwd_return_5d", "fwd_inflows_5d") if c in built.frame.columns]
    X, y = make_feature_frame(built.frame, spec, extra_drop=extra)
    X = X.drop(columns=[c for c in X.columns
                        if pd.api.types.is_datetime64_any_dtype(X[c])], errors="ignore")
    matrices[key] = (X, built.frame[built.target])

DATA_HASHES = {r.path.name: r.sha256 for r in results.values()}
pd.DataFrame([{"module": k, "rows": len(X), "features": X.shape[1]}
              for k, (X, y) in matrices.items()])

## 2 · Explainability

SHAP assigns each feature its average marginal contribution across all feature
orderings — the Shapley value. It is the only attribution method with a
uniqueness guarantee under its axioms.

**The caveat to state before an examiner does:** SHAP explains the *model*, not
the world. A high attribution for `Interest_Rate` does not mean raising rates
causes default — it means the incumbent pricing already encodes risk, which is
exactly what leakage-register entry **L-06** records.

In [ ]:
from novafin.models import ModelSpec
from novafin.models.build import build_pipeline

explanations = {}
for module in ("loans", "transactions", "initiatives"):
    X, y = matrices[module]
    catalog = load_model_specs(module)
    spec = catalog.get("lightgbm") if "lightgbm" in catalog.names() else catalog.specs[-1]
    pipeline = build_pipeline(spec, X, cfg=cfg, defaults=catalog.defaults)
    pipeline.fit(X, np.asarray(y))

    numeric = X.select_dtypes(include=[np.number])
    explanation = compute_shap(
        pipeline.named_steps["model"] if hasattr(pipeline, "named_steps") else pipeline,
        numeric, module=module, model_name=spec.name, max_samples=2000,
        seed=cfg.reproducibility.seed,
    )
    explanations[module] = explanation
    print(f"{module:<14} method = {explanation.method}")
    for note in explanation.notes:
        print("   note:", note)

display(global_importance(list(explanations.values()), top=8))

In [ ]:
# Per-applicant attribution - the "why was I declined?" answer that makes a
# lending model deployable.
credit_explanation = explanations["loans"]
if credit_explanation.values is not None:
    instance = explain_instance(credit_explanation, index=0, top=8)
    display(instance)
    print(f"base value (average prediction): {instance.attrs['base_value']:.4f}")
    print(f"this applicant's prediction    : {instance.attrs['prediction']:.4f}")

    fig, ax = plt.subplots(figsize=(8, 4))
    colours = [semantic_color("critical") if c > 0 else color("teal")
               for c in instance["contribution"]]
    ax.barh(instance["feature"], instance["contribution"], color=colours)
    ax.axvline(0, color=color("silver"), lw=0.8)
    ax.invert_yaxis()
    ax.set_xlabel("SHAP contribution")
    ax.set_title("M2 — why this applicant received this score")
    save_figure(fig, "06_m02_instance_explanation", close=False); plt.show()

In [ ]:
# Model cards - the governance artefact. Limitations and leakage controls are
# mandatory fields, populated from findings the repository actually established.
cards = []
for module, explanation in explanations.items():
    X, y = matrices[module]
    catalog = load_model_specs(module)
    spec = catalog.get("lightgbm") if "lightgbm" in catalog.names() else catalog.specs[-1]
    splitter, split_kwargs = make_splitter(module, features[module].frame, cfg=cfg)
    cv = cross_validate_model(spec, X, y, splitter, task=cfg.dataset(module).task,
                              module=module, cfg=cfg, split_kwargs=split_kwargs,
                              defaults=catalog.defaults)
    evaluation = evaluate_cv(cv, y, cfg=cfg)
    card = build_model_card(module, spec.name, cfg.dataset(module).task,
                            evaluation.metrics, explanation=explanation, cfg=cfg,
                            data_hashes=DATA_HASHES)
    cards.append(card)

(cfg.paths.reports / "model_cards.md").write_text(
    "\n\n---\n\n".join(c.to_markdown() for c in cards), encoding="utf-8"
)
print("model cards written to reports/model_cards.md")
print(cards[0].to_markdown()[:1800])

## 3 · Calibration — the metric that decides whether ECL is real

A model can rank perfectly (AUC 1.0) and still be useless for `ECL = PD × LGD ×
EAD` if it reports 0.30 where the truth is 0.20. Expected NPV has the same
dependency. So Brier and ECE are reported beside discrimination, never instead
of it.

In [ ]:
X_loans, y_loans = matrices["loans"]
splitter_loans, kwargs_loans = make_splitter("loans", features["loans"].frame, cfg=cfg)
catalog_loans = load_model_specs("loans")

calibration_rows = []
for spec in catalog_loans.specs:
    if spec.unsupervised:
        continue
    cv = cross_validate_model(spec, X_loans, y_loans, splitter_loans,
                              task="binary_classification", module="loans", cfg=cfg,
                              split_kwargs=kwargs_loans, defaults=catalog_loans.defaults)
    evaluation = evaluate_cv(cv, y_loans, cfg=cfg)
    mask = cv.oof_mask
    calibration_rows.append({
        "model": spec.name,
        "roc_auc": evaluation.metrics.get("roc_auc"),
        "ks": evaluation.metrics.get("ks"),
        "brier": evaluation.metrics.get("brier"),
        "ece": expected_calibration_error(np.asarray(y_loans)[mask],
                                          np.asarray(cv.oof_predictions)[mask]),
    })

calibration = pd.DataFrame(calibration_rows).round(5)
display(calibration)
print("A model can win on KS and lose on ECE. ECL multiplies the PROBABILITY by")
print("money, so ECE is what decides whether the expected-loss numbers are usable.")

## 4 · The fraud holdout — scored once

2026-06-01 to 2026-08-31 has been untouched since Phase 2. Every tuning
decision has now been made, so it is scored **once**. Scoring it earlier — even
"just to look" — would have made it a validation set.

In [ ]:
X_txn, y_txn = matrices["transactions"]
txn_frame = features["transactions"].frame
train_idx, holdout_idx = holdout_by_time(txn_frame, "transactions", cfg=cfg)
splitter_txn, _ = make_splitter("transactions", txn_frame, cfg=cfg)
catalog_txn = load_model_specs("transactions")
spec_txn = catalog_txn.get("lightgbm")

# 1. Fit on training only; find the cost-optimal threshold out of fold.
cv_txn = cross_validate_model(spec_txn, X_txn.iloc[train_idx], y_txn.iloc[train_idx],
                              splitter_txn, task="binary_classification",
                              module="transactions", cfg=cfg, defaults=catalog_txn.defaults)
mask = cv_txn.oof_mask
cost_fn = cfg.fin("fraud", "cost_missed_fraud_inr", default=10000)
cost_fp = cfg.fin("fraud", "cost_false_positive_inr", default=500)
best = optimal_threshold(np.asarray(y_txn.iloc[train_idx])[mask],
                         np.asarray(cv_txn.oof_predictions)[mask],
                         cost_false_negative=cost_fn, cost_false_positive=cost_fp)
print("threshold chosen on TRAINING data only:", round(best["threshold"], 4))

# 2. Refit on all training data, score the holdout ONCE.
pipeline = build_pipeline(spec_txn, X_txn.iloc[train_idx], cfg=cfg, defaults=catalog_txn.defaults)
pipeline.fit(X_txn.iloc[train_idx], np.asarray(y_txn.iloc[train_idx]))
holdout_scores = pipeline.predict_proba(X_txn.iloc[holdout_idx])[:, 1]
holdout_truth = np.asarray(y_txn.iloc[holdout_idx])

holdout_metrics = {
    "roc_auc": roc_auc(holdout_truth, holdout_scores),
    "pr_auc": average_precision(holdout_truth, holdout_scores),
    "base_rate": float(holdout_truth.mean()),
    "n": len(holdout_truth),
}
print("\nHOLDOUT (2026-06-01 to 2026-08-31), scored ONCE:")
for name, value in holdout_metrics.items():
    print(f"   {name:<12} {value:.5f}" if isinstance(value, float) else f"   {name:<12} {value}")

train_scores = np.asarray(cv_txn.oof_predictions)[mask]
psi = population_stability_index(train_scores, holdout_scores)
print(f"\nPSI (train vs holdout score distribution): {psi:.4f}")
print("   <0.10 stable | 0.10-0.25 monitor | >0.25 population has shifted")

In [ ]:
# Apply the training-chosen threshold to the holdout. THIS is the honest
# estimate of what the system would have cost in production.
flagged = holdout_scores >= best["threshold"]
tp = int((flagged & (holdout_truth == 1)).sum())
fp = int((flagged & (holdout_truth == 0)).sum())
fn = int((~flagged & (holdout_truth == 1)).sum())
holdout_cost = fn * cost_fn + fp * cost_fp

print(f"flagged {flagged.sum():,} of {len(flagged):,} transactions")
print(f"   caught {tp} frauds, missed {fn}, investigated {fp} false positives")
print(f"   realised cost: ₹{holdout_cost:,.0f}")
print(f"   cost of flagging nothing: ₹{holdout_truth.sum() * cost_fn:,.0f}")
print(f"   saving: ₹{holdout_truth.sum() * cost_fn - holdout_cost:,.0f}")

## 5 · Market risk — VaR and expected shortfall

Three VaR methods are reported because the gaps between them are themselves a
finding: the parametric figure assumes normality, and the distance from the
historical one quantifies how fat the tails really are.

Expected shortfall is reported alongside because VaR is **not sub-additive** —
two portfolios can have a combined VaR larger than the sum of their parts,
which is incoherent for a risk measure. Basel III (FRTB) moved to ES for that
reason.

In [ ]:
market_frame = features["market"].frame
returns_panel = market_frame.pivot_table(index="Date", columns="Ticker", values="Return")
portfolio_returns = returns_panel.mean(axis=1).dropna()

var_rows = []
for confidence in cfg.fin("market_risk", "var_confidence", default=[0.95, 0.99]):
    var_rows.append({
        "confidence": confidence,
        "historical": historical_var(portfolio_returns, confidence),
        "parametric": parametric_var(portfolio_returns, confidence),
        "monte_carlo": monte_carlo_var(portfolio_returns, confidence,
                                       seed=cfg.reproducibility.seed),
        "expected_shortfall": expected_shortfall(portfolio_returns, confidence),
    })
var_table = pd.DataFrame(var_rows).round(5)
display(var_table)

fat_tail_gap = var_table.loc[var_table.confidence == 0.99, "historical"].iloc[0] - \
               var_table.loc[var_table.confidence == 0.99, "parametric"].iloc[0]
print(f"99% historical minus parametric: {fat_tail_gap:+.5f}")
print("A POSITIVE gap means the empirical tail is fatter than a normal assumes.")

In [ ]:
# Kupiec proportion-of-failures backtest. A 99% VaR should be breached on
# about 1% of days - too few means idle capital, too many means understated risk.
kupiec_rows = []
for confidence in (0.95, 0.99):
    var = historical_var(portfolio_returns, confidence)
    exceptions = int((portfolio_returns < -var).sum())
    result = kupiec_pof_test(exceptions, len(portfolio_returns), confidence)
    kupiec_rows.append({"confidence": confidence, "var": var, **result})

kupiec = pd.DataFrame(kupiec_rows).round(5)
display(kupiec[["confidence", "var", "exceptions", "n_observations",
                "observed_rate", "expected_rate", "lr_statistic", "p_value"]])
print("p > 0.05 means the VaR model is NOT rejected - the breach count is")
print("consistent with the stated confidence level.")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(portfolio_returns.index, portfolio_returns, lw=0.7, color=color("steel"), label="daily return")
for confidence, style in ((0.95, "--"), (0.99, ":")):
    level = -historical_var(portfolio_returns, confidence)
    ax.axhline(level, ls=style, lw=1.4, color=semantic_color("critical"),
               label=f"{confidence:.0%} VaR")
breaches = portfolio_returns[portfolio_returns < -historical_var(portfolio_returns, 0.99)]
ax.scatter(breaches.index, breaches, s=22, color=semantic_color("warning"),
           zorder=5, label="99% breaches")
ax.legend(ncol=4, fontsize=8)
ax.set_title("Equity portfolio — VaR backtest")
save_figure(fig, "06_var_backtest", close=False); plt.show()

## 6 · Portfolio backtest — ML vs equal-weight vs Markowitz

1/N is not a straw man. DeMiguel, Garlappi & Uppal (2009) showed it beats most
optimised portfolios out of sample, because estimation error in the covariance
matrix swamps the optimisation gain. **If the ML portfolio does not beat 1/N
after costs, that is the result we report.**

In [ ]:
# Build the signal panel from out-of-fold predictions on the equity panel.
X_mkt, y_mkt = matrices["market"]
splitter_mkt, kwargs_mkt = make_splitter("market", market_frame, cfg=cfg)
catalog_mkt = load_model_specs("market")
spec_mkt = catalog_mkt.get("lightgbm")

cv_mkt = cross_validate_model(spec_mkt, X_mkt, y_mkt, splitter_mkt,
                              task="panel_regression", module="market", cfg=cfg,
                              split_kwargs=kwargs_mkt, defaults=catalog_mkt.defaults,
                              ic_groups=market_frame["Date"])
signal_frame = market_frame[["Date", "Ticker"]].copy()
signal_frame["prediction"] = cv_mkt.oof_predictions
signal_panel = signal_frame.pivot_table(index="Date", columns="Ticker", values="prediction")
returns_aligned = returns_panel.loc[signal_panel.index, signal_panel.columns]

cost_bps = cfg.fin("equity", "transaction_cost_bps", default=10)
strategies = {}
for name in ("equal", "markowitz", "ml"):
    series, weights = backtest_portfolio(
        returns_aligned, signal_panel if name == "ml" else None,
        strategy=name, rebalance=cfg.fin("equity", "rebalance_frequency", default="ME"),
        long_quantile=cfg.fin("equity", "long_quantile", default=0.2),
        transaction_cost_bps=cost_bps,
        risk_free_rate=cfg.fin("equity", "risk_free_rate_annual", default=0.06),
    )
    strategies[name] = series

comparison = pd.DataFrame({
    name: portfolio_metrics(series,
                            risk_free_rate=cfg.fin("equity", "risk_free_rate_annual", default=0.06))
    for name, series in strategies.items()
}).T.round(4)
display(comparison)
print(f"\n(net of {cost_bps} bps per side on the turnover actually traded)")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
palette = {"equal": color("beige"), "markowitz": color("steel"), "ml": color("teal")}
for name, series in strategies.items():
    ax.plot(series.index, (1 + series).cumprod(), lw=2,
            color=palette[name], label=f"{name} (Sharpe {comparison.loc[name, 'sharpe']:.2f})")
ax.set_ylabel("Growth of ₹1"); ax.set_title("Portfolio backtest, net of transaction costs")
ax.legend()
save_figure(fig, "06_portfolio_backtest", close=False); plt.show()

best_strategy = comparison["sharpe"].idxmax()
print(f"highest Sharpe: {best_strategy}")
if best_strategy != "ml":
    print("\nThe ML portfolio did NOT win. Report that. DeMiguel et al. (2009) found")
    print("1/N beats most optimised portfolios out of sample, and a capstone that")
    print("reproduces a known result honestly is worth more than one that hides it.")

## 7 · Liquidity stress testing

The reserve floor in Module 11 is not a preference — it is whatever survives
the severe scenario. Two mechanisms are applied jointly because that is how a
liquidity crisis actually happens: depositors withdraw *and* wholesale funding
fails to roll, at the same time.

In [ ]:
liq = features["liquidity"].frame.iloc[-1]
scenarios = cfg.fin("liquidity_risk", "stress_scenarios", default={})

stress_rows = []
for name, parameters in scenarios.items():
    outcome = stress_test_liquidity(
        deposits=float(liq["Deposits"]),
        wholesale_funding=float(liq["Wholesale_Funding"]),
        liquid_assets=float(liq["Cash"] + liq["Securities_Holdings"]),
        expected_outflows=float(liq["Expected_Outflows"]),
        deposit_runoff=float(parameters["deposit_runoff"]),
        funding_haircut=float(parameters["funding_haircut"]),
    )
    stress_rows.append({"scenario": name, **outcome})

stress = pd.DataFrame(stress_rows).round(2)
display(stress[["scenario", "deposit_runoff", "funding_haircut", "stressed_outflows",
                "surviving_buffer", "coverage_ratio", "survives"]])

worst = stress.loc[stress["coverage_ratio"].idxmin()]
print(f"\nWorst scenario: {worst['scenario']} -> coverage {worst['coverage_ratio']:.2f}x")
print("The Module 11 liquidity floor is set by THIS number, not by preference.")

## 8 · Module 11 — the ₹1,000 crore allocation

Every module now contributes a number. This is where the brief's chain
terminates.

> **Honest constraint:** the customer file shares **zero** `Customer_ID` values
> with the loan file (finding **N-02**), so the integration runs at
> portfolio/segment level, not customer level.

In [ ]:
# Feed each module's MEASURED output into the bucket inputs. Anything left
# unsupplied falls back to a labelled placeholder, and the result records that.
loans_frame = features["loans"].frame
cv_loans = cross_validate_model(catalog_loans.get("lightgbm"), X_loans, y_loans,
                                splitter_loans, task="binary_classification",
                                module="loans", cfg=cfg, split_kwargs=kwargs_loans,
                                defaults=catalog_loans.defaults)
pd_oof = pd.Series(cv_loans.oof_predictions, index=loans_frame.index)
lgd = cfg.fin("credit", "lgd", default=0.40)
ecl = expected_credit_loss(pd_oof, loans_frame["Loan_Amount"], lgd=lgd)

is_corporate = loans_frame["Customer_Type"] == "Corporate"
credit_metrics = {
    "corporate_yield": float(loans_frame.loc[is_corporate, "Interest_Rate"].mean() / 100),
    "retail_yield": float(loans_frame.loc[~is_corporate, "Interest_Rate"].mean() / 100),
    "corporate_ecl_rate": float(ecl[is_corporate].sum() / loans_frame.loc[is_corporate, "Loan_Amount"].sum()),
    "retail_ecl_rate": float(ecl[~is_corporate].sum() / loans_frame.loc[~is_corporate, "Loan_Amount"].sum()),
    "corporate_volatility": 0.045, "retail_volatility": 0.038,
    "source": "M2 PD model, out-of-fold, notebook 06",
}
equity_metrics = {
    "annual_return": float(comparison.loc[best_strategy, "annual_return"]),
    "annual_volatility": float(comparison.loc[best_strategy, "annual_volatility"]),
    "source": f"M5/M6 backtest ({best_strategy}), notebook 06",
}
fraud_metrics = {
    "fraud_drag": float(holdout_cost / max(X_txn.iloc[holdout_idx]["Amount"].sum(), 1)),
    "source": "M3 holdout realised cost, notebook 06",
}
liquidity_metrics = {
    "min_reserve_pct": float(max(0.10, 1 / max(worst["coverage_ratio"], 1.0) * 0.25)),
    "reserve_yield": 0.055,
    "source": "M7 severe stress scenario, notebook 06",
}

print("MEASURED inputs:")
for name, block in [("credit", credit_metrics), ("equity", equity_metrics),
                    ("fraud", fraud_metrics), ("liquidity", liquidity_metrics)]:
    print(f"  {name}: " + ", ".join(
        f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in block.items()))

In [ ]:
buckets = build_bucket_inputs(
    cfg, credit_metrics=credit_metrics, equity_metrics=equity_metrics,
    fraud_metrics=fraud_metrics, liquidity_metrics=liquidity_metrics,
)
allocation = allocate_capital(buckets, cfg=cfg, risk_aversion=2.0,
                              max_portfolio_volatility=0.08)

display(allocation.allocations[
    ["bucket", "allocation_crore", "allocation_pct", "net_return", "volatility",
     "risk_adjusted_return", "expected_profit_crore"]
].round(4))

print("\nSUMMARY")
for name, value in allocation.summary().items():
    print(f"   {name:<28} {value:,.2f}")
print("\nBINDING CONSTRAINTS")
for item in allocation.constraints_binding:
    print("   -", item)
print("\nLIMITATIONS")
for item in allocation.limitations:
    print("   -", item)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
frame = allocation.allocations.sort_values("allocation_crore", ascending=True)
axes[0].barh(frame["bucket"].str.replace("_", " "), frame["allocation_crore"],
             color=[semantic_color("critical") if v < 0 else color("teal")
                    for v in frame["net_return"]])
axes[0].set_xlabel("₹ crore"); axes[0].set_title("Recommended deployment of ₹1,000 crore")

axes[1].scatter(allocation.allocations["volatility"] * 100,
                allocation.allocations["net_return"] * 100,
                s=allocation.allocations["allocation_crore"] * 1.5,
                color=color("teal"), alpha=0.75, edgecolor="white")
for _, row in allocation.allocations.iterrows():
    axes[1].annotate(row["bucket"].replace("_", " "),
                     (row["volatility"] * 100, row["net_return"] * 100),
                     fontsize=7, xytext=(4, 4), textcoords="offset points")
axes[1].set_xlabel("Volatility (%)"); axes[1].set_ylabel("Net return (%)")
axes[1].set_title("Risk vs return (bubble = allocation)")
plt.tight_layout()
save_figure(fig, "06_capital_allocation", close=False); plt.show()

In [ ]:
sensitivity = sensitivity_analysis(buckets, cfg=cfg)
display(sensitivity)
print("If the allocation barely moves across risk aversion, the CONSTRAINTS -")
print("not the optimiser - are driving it. The result object says so explicitly,")
print("and that is a finding to report rather than hide.")

narrative = allocation_narrative(allocation)
(cfg.paths.reports / "allocation_recommendation.md").write_text(narrative, encoding="utf-8")
print("\nwritten: reports/allocation_recommendation.md")
print(narrative[:2500])

## 9 · Persist

In [ ]:
cfg.paths.tables.mkdir(parents=True, exist_ok=True)
calibration.to_csv(cfg.paths.tables / "06_calibration.csv", index=False)
var_table.to_csv(cfg.paths.tables / "06_var.csv", index=False)
kupiec.to_csv(cfg.paths.tables / "06_kupiec.csv", index=False)
comparison.to_csv(cfg.paths.tables / "06_portfolio_comparison.csv")
stress.to_csv(cfg.paths.tables / "06_liquidity_stress.csv", index=False)
allocation.allocations.to_csv(cfg.paths.tables / "06_capital_allocation.csv", index=False)
sensitivity.to_csv(cfg.paths.tables / "06_allocation_sensitivity.csv", index=False)
global_importance(list(explanations.values()), top=15).to_csv(
    cfg.paths.tables / "06_shap_importance.csv", index=False)
pd.DataFrame([holdout_metrics]).to_csv(cfg.paths.tables / "06_fraud_holdout.csv", index=False)

print("Tables written:")
for path in sorted(cfg.paths.tables.glob("06_*.csv")):
    print("   ", path.name)

---

## Phase 7 summary

| Deliverable | Status |
|---|---|
| SHAP + per-instance attribution | ✅ |
| Model cards with mandatory limitations | ✅ `reports/model_cards.md` |
| Calibration beside discrimination | ✅ |
| Fraud holdout **scored once** | ✅ |
| VaR ×3 methods + ES + Kupiec backtest | ✅ |
| Portfolio backtest vs 1/N and Markowitz | ✅ |
| Liquidity stress → the reserve floor | ✅ |
| **Module 11 ₹1,000 cr allocation** | ✅ `reports/allocation_recommendation.md` |

**NEXT:** `07_inference_demo` — scoring new rows from a saved bundle, and the
Gradio app.